In [ ]:
# 若没装 fastkaggle 就安装，然后导入
# install fastkaggle if not available
try: import fastkaggle
except ModuleNotFoundError:
    !pip install -Uq fastkaggle

from fastkaggle import *

In [ ]:
# 设置竞赛并下载数据（会一并装 fastai / timm）
comp = 'paddy-disease-classification'

path = setup_comp(comp, install='fastai "timm>=0.6.2.dev0"')

In [ ]:
# 看数据路径
path

In [ ]:
# 导入 fastai 视觉模块，固定种子，看目录结构
from fastai.vision.all import *
set_seed(42)

path.ls()

In [ ]:
# 训练图片路径 + 取所有图片文件
trn_path = path/'train_images'
files = get_image_files(trn_path)

In [ ]:
# 打开第一张图看尺寸和缩略图
img = PILImage.create(files[0])
print(img.size)
img.to_thumb(128)

In [ ]:
# 并行统计所有图片的尺寸分布
from fastcore.parallel import *

def f(o): return PILImage.create(o).size
sizes = parallel(f, files, n_workers=8)
pd.Series(sizes).value_counts()

In [ ]:
# 建 DataLoaders：先 squish 到 480，再做增强并缩到 128
dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, seed=42,
    item_tfms=Resize(480, method='squish'),
    batch_tfms=aug_transforms(size=128, min_scale=0.75))

dls.show_batch(max_n=6)

In [ ]:
# 用 resnet26d 建模型（半精度 fp16 加速）
learn = vision_learner(dls, 'resnet26d', metrics=error_rate, path='.').to_fp16()

In [ ]:
# 找学习率
learn.lr_find(suggest_funcs=(valley, slide))

In [ ]:
# 微调 3 轮，lr=0.01
learn.fine_tune(3, 0.01)

In [ ]:
# 读提交模板 sample_submission
ss = pd.read_csv(path/'sample_submission.csv')
ss

In [ ]:
# 取测试图片，包成 test dataloader
tst_files = get_image_files(path/'test_images').sorted()
tst_dl = dls.test_dl(tst_files)

In [ ]:
# 预测，拿到解码后的类别索引
probs,_,idxs = learn.get_preds(dl=tst_dl, with_decoded=True)
idxs

In [ ]:
# 看类别词表 vocab
dls.vocab

In [ ]:
# 把索引映射回类别名
mapping = dict(enumerate(dls.vocab))
results = pd.Series(idxs.numpy(), name="idxs").map(mapping)
results

In [ ]:
# 写入提交文件并查看前几行
ss['label'] = results
ss.to_csv('subm.csv', index=False)
!head subm.csv

In [ ]:
# 提交到 Kaggle（仅非 Kaggle 环境）
if not iskaggle:
    from kaggle import api
    api.competition_submit_cli('subm.csv', 'initial rn26d 128px', comp)

In [ ]:
# 把 notebook 推送到 Kaggle（Jeremy 自用）
if not iskaggle:
    push_notebook('jhoward', 'first-steps-road-to-the-top-part-1',
                  title='First Steps: Road to the Top, Part 1',
                  file='first-steps-road-to-the-top-part-1.ipynb',
                  competition=comp, private=False, gpu=True)